# Hi-EF Stage A: train-only conditional-attribution audit

This notebook runs the frozen cross-fitted audit for Party A's incremental, context-dependent predictive utility. It reads only rows whose original split is `train`; original validation and test are not evaluated.

Attach `ptrnghieu/hi-ef-features-v2`, enable **Internet** and a **T4 GPU**. First run in `smoke` mode. Only after it succeeds, change `RUN_MODE` to `full` and use **Save Version → Save & Run All**.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/train_conditional_attribution_audit')

if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)

assert (FEATURES / '01_00059.pt').is_file(), 'Attach ptrnghieu/hi-ef-features-v2'
assert (REPO / 'experiments/RESEARCH_SPEC_v1.0.md').is_file()
assert (REPO / 'experiments/run_train_conditional_attribution_audit.py').is_file()
print('Repository and features ready at', REPO)

In [ ]:
# Keep `smoke` for the first successful run.
# Change exactly this value to `full` for the research audit.
RUN_MODE = 'smoke'  # 'smoke' or 'full'

if RUN_MODE == 'smoke':
    run_args = [
        '--seeds', '42', '--epochs', '1',
        '--replacement-replicates', '1', '--bootstrap-replicates', '100',
    ]
    OUTPUT = OUTPUT.with_name(OUTPUT.name + '_smoke')
elif RUN_MODE == 'full':
    run_args = []  # frozen defaults: 5 seeds, 4 folds, 8 epochs, 10 replacements, 5,000 bootstrap draws
else:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

print('Mode:', RUN_MODE)
print('Output:', OUTPUT)

In [ ]:
command = [
    'python', str(REPO / 'experiments/run_train_conditional_attribution_audit.py'),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--features-dir', str(FEATURES),
    '--output-dir', str(OUTPUT),
    '--batch-size', '32', '--workers', '2',
] + run_args
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd

summary = json.loads((OUTPUT / 'audit_summary.json').read_text())
assert summary['original_partition_read'] == 'train'
assert summary['validation_evaluated'] is False
assert summary['test_evaluated'] is False
print(json.dumps(summary['actual_A_vs_matched_ranking'], indent=2))
print(json.dumps(summary['h3_advancement_gate'], indent=2))
display(pd.read_csv(OUTPUT / 'overall_metrics.csv'))
display(pd.read_csv(OUTPUT / 'ranking_strata.csv'))

## Preserve these outputs

For the full run, save the entire output directory. The files needed for the follow-up analysis are `audit_summary.json`, `overall_metrics.csv`, `stratified_metrics.csv`, `ranking_strata.csv`, `true_vs_matched_ranking.csv`, and `crossfit_candidate_predictions.csv`. Smoke-mode numbers are only a pipeline check and must not be interpreted as research results.